# Face Attendance Prototype

This notebook creates a face-embedding gallery from `data/registered_students` and processes recorded videos in `test_videos`. It is enrollment using a pre-trained model, not training a neural network from scratch.

Only use images and videos collected with permission.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT = Path.cwd()
if not (PROJECT / 'attendance_pipeline.py').exists():
    PROJECT = Path(r'C:/Users/akash/attendance_project')
sys.path.insert(0, str(PROJECT))
import attendance_pipeline as pipeline

print('Project:', PROJECT)
print('Student folders:', [p.name for p in (PROJECT / 'data' / 'registered_students').iterdir() if p.is_dir()])
print('Videos:', [p.name for p in (PROJECT / 'test_videos').glob('*.mp4')])

Project: C:\Users\akash\attendance_project
Student folders: ['person_001', 'person_002']
Videos: ['test1.mp4', 'test2.mp4', 'test3.mp4', 'test4.mp4']


## Build the face gallery

Run this cell after adding or changing registration images. The first run downloads the two ONNX model files.

In [2]:
gallery = pipeline.build_gallery()
print('Registered IDs:', list(gallery['labels']))
print('Gallery shape:', gallery['features'].shape)

Models are ready.
person_001: 14 usable images
person_002: 16 usable images
Saved gallery: C:\Users\akash\attendance_project\models\gallery.npz
Registered IDs: [np.str_('person_001'), np.str_('person_002')]
Gallery shape: (2, 128)


## Process one recorded video

Change `VIDEO_NAME` to `test1.mp4`, `test2.mp4`, `test3.mp4`, or `test4.mp4`.

In [3]:
VIDEO_NAME = 'test1.mp4'
video_path = PROJECT / 'test_videos' / VIDEO_NAME
processed_path = pipeline.process_video(video_path)
print('Processed video:', processed_path)

Models are ready.
{
  "video": "C:\\Users\\akash\\attendance_project\\test_videos\\test1.mp4",
  "processed_video": "C:\\Users\\akash\\attendance_project\\outputs\\test1_processed.mp4",
  "attendance_csv": "C:\\Users\\akash\\attendance_project\\outputs\\test1_attendance.csv",
  "present": {
    "person_002": {
      "frame": 19,
      "time_seconds": 0.64,
      "score": 0.8005
    },
    "person_001": {
      "frame": 34,
      "time_seconds": 1.14,
      "score": 0.8616
    }
  }
}
Processed video: C:\Users\akash\attendance_project\outputs\test1_processed.mp4


## View the attendance CSV

In [4]:
attendance_csv = PROJECT / 'outputs' / f'{Path(VIDEO_NAME).stem}_attendance.csv'
pd.read_csv(attendance_csv)

,student_id,status,first_confirmed_seconds,similarity,processed_at
0,person_001,Present,1.14,0.8616,2026-08-30T12:03:40
1,person_002,Present,0.64,0.8005,2026-08-30T12:03:40


## Tuning

If known faces remain red, try lowering `pipeline.COSINE_THRESHOLD` gradually, for example to `0.42`. If the system confuses people, increase it, for example to `0.50`, and collect better images. Do not claim 100% certainty; report the threshold and test results.

This initial version draws face boxes. Full-body red boxes when faces are too far away require a separate person detector and can be added later.